### Machine Learning Model: Regression
##### TASK: Build a model to predict the price of a car

#### install & import libraries

In [1]:
import sys
!{sys.executable} -m pip install matplotlib seaborn scikit-learn pandas numpy --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

#### load data

In [ ]:
car_df = pd.read_csv("car_price_prediction.csv")
car_df.head()

,ID,Price,Levy,Manufacturer,Model,Prod. year,Category,Leather interior,Fuel type,Engine volume,Mileage,Cylinders,Gear box type,Drive wheels,Doors,Wheel,Color,Airbags
0,45654403,13328,1399,LEXUS,RX 450,2010,Jeep,Yes,Hybrid,3.5,186005 km,6,Automatic,4x4,4-May,Left wheel,Silver,12
1,44731507,16621,1018,CHEVROLET,Equinox,2011,Jeep,No,Petrol,3,192000 km,6,Tiptronic,4x4,4-May,Left wheel,Black,8
2,45774419,8467,-,HONDA,FIT,2006,Hatchback,No,Petrol,1.3,200000 km,4,Variator,Front,4-May,Right-hand drive,Black,2
3,45769185,3607,862,FORD,Escape,2011,Jeep,Yes,Hybrid,2.5,168966 km,4,Automatic,4x4,4-May,Left wheel,White,0
4,45809263,11726,446,HONDA,FIT,2014,Hatchback,Yes,Petrol,1.3,91901 km,4,Automatic,Front,4-May,Left wheel,Silver,4


In [4]:
car_df.columns

Index(['ID', 'Price', 'Levy', 'Manufacturer', 'Model', 'Prod. year',
       'Category', 'Leather interior', 'Fuel type', 'Engine volume', 'Mileage',
       'Cylinders', 'Gear box type', 'Drive wheels', 'Doors', 'Wheel', 'Color',
       'Airbags'],
      dtype='object')

#### clean data

In [5]:
# cast price to numeric, replace '-' levy with 0
car_df['price'] = pd.to_numeric(car_df['price'], errors='coerce')
car_df['levy'] = car_df['levy'].str.replace('-', '0', regex=False)

# strip 'km' and any spaces/commas from mileage, then cast
car_df['mileage'] = (
    car_df['mileage']
    .str.replace('km', '', regex=False)
    .str.replace(' ', '', regex=False)
    .str.replace(',', '', regex=False)
    .str.strip()
)
car_df['mileage'] = pd.to_numeric(car_df['mileage'], errors='coerce')

# strip 'Turbo' and whitespace from engine_volume, then cast
car_df['engine_volume'] = (
    car_df['engine_volume']
    .str.replace('Turbo', '', regex=False)
    .str.strip()
)
car_df = car_df.astype({'levy': float, 'engine_volume': float})

# rename columns
car_df = car_df.rename(columns={
    'Price': 'price',
    'Levy': 'levy',
    'Model': 'model',
    'Prod. year': 'production_year',
    'Fuel type': 'fuel_type',
    'Engine volume': 'engine_volume',
    'Mileage': 'mileage',
    'Cylinders': 'cylinders',
    'Airbags': 'airbags',
    'Manufacturer': 'manufacturer'
})

# drop rows where key columns couldn't be parsed
car_df.dropna(subset=['price', 'levy', 'mileage', 'engine_volume'], inplace=True)

car_df.head()

KeyError: 'price'

In [ ]:
car_df.info()

#### visualize data

In [ ]:
# price vs levy
plt.scatter(car_df['levy'], car_df['price'])
plt.xlabel("Levy")
plt.ylabel("Price")
plt.title("Price vs Levy")
plt.show()

In [ ]:
# price vs production year
plt.scatter(car_df['production_year'], car_df['price'])
plt.xlabel("Production Year")
plt.ylabel("Price")
plt.title("Price vs Production Year")
plt.show()

In [ ]:
# price vs mileage
plt.scatter(car_df['mileage'], car_df['price'])
plt.xlabel("Mileage")
plt.ylabel("Price")
plt.title("Price vs Mileage")
plt.show()

In [ ]:
# price vs cylinders
plt.scatter(car_df['cylinders'], car_df['price'])
plt.xlabel("Cylinders")
plt.ylabel("Price")
plt.title("Price vs Cylinders")
plt.show()

In [ ]:
# price vs airbags (with regression line)
sns.lmplot(x='airbags', y='price', data=car_df, aspect=1.5, scatter_kws={'alpha': 0.2})
plt.show()

#### select features and label

In [ ]:
feature_cols = ['production_year', 'levy', 'mileage', 'cylinders', 'airbags']

X = car_df[feature_cols]
y = car_df['price']  # 1-D Series

### model data

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

# scale features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

# fit model
linreg = LinearRegression().fit(X_train, y_train)
linreg

In [ ]:
# display coefficients
print("Intercept:    {}".format(linreg.intercept_))
print("Coefficients: {}".format(linreg.coef_))

In [ ]:
# make predictions on test data
y_pred = linreg.predict(X_test)
y_pred

In [ ]:
# evaluate model
print(f"R\u00b2 Score: {r2_score(y_test, y_pred):.3f}")
print(f"MSE:      {mean_squared_error(y_test, y_pred):.3f}")
print(f"RMSE:     {np.sqrt(mean_squared_error(y_test, y_pred)):.3f}")

#### save model

In [ ]:
import pickle

# save model
with open("model.pkl", "wb") as f:
    pickle.dump(linreg, f)

# save scaler (must be applied to inputs in your app too)
with open("scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

# save feature names
with open("features.txt", "w") as f:
    f.write("\n".join(feature_cols))

print("Saved: model.pkl, scaler.pkl, features.txt")